# NPCI UPI Ecosystem Reliability & Technical Decline Audit

### Statistical & Root-Cause Diagnostics.

In [1]:
import pandas as pd
import scipy.stats as stats
import numpy as np

# Load clean CSV dataset
df = pd.read_csv(r'D:\DATA_ANALYST_PROJECTS_2026\End_to_End_DATA_ANALYST_PROJECTS\NPCI UPI Ecosystem Reliability & Technical Decline Audit\data\npci_upi_clean_data.csv')


In [2]:

print("==================================================")
print("=== 1. VOLUME STRESS CORRELATION (Top 10 Banks) ===")
print("==================================================")
# Group by volume to find the top 10 busiest banks in the ecosystem
top_banks = df.groupby('bank_name')['total_volume_mn'].sum().nlargest(10).index

for bank in top_banks:
    bank_data = df[(df['bank_name'] == bank) & (df['role_type'] == 'Remitter')]
    if len(bank_data) > 5:
        # Calculate Pearson correlation between total volume and technical decline percentage
        corr, p_value = stats.pearsonr(bank_data['total_volume_mn'], bank_data['technical_decline_pct'])
        print(f"{bank}: Pearson r = {corr:.2f}, p-value = {p_value:.4f}")


=== 1. VOLUME STRESS CORRELATION (Top 10 Banks) ===
Yes Bank Ltd.: Pearson r = -0.19, p-value = 0.3927
State Bank of India: Pearson r = 0.18, p-value = 0.4666
State Bank Of India: Pearson r = -0.45, p-value = 0.1447
Axis Bank Ltd.: Pearson r = -0.47, p-value = 0.0107
HDFC Bank Ltd.: Pearson r = -0.16, p-value = 0.5017
Bank of Baroda: Pearson r = -0.21, p-value = 0.2853
Canara Bank: Pearson r = -0.36, p-value = 0.0488
Punjab National Bank: Pearson r = 0.35, p-value = 0.0703
Union Bank of India: Pearson r = 0.17, p-value = 0.4261


In [3]:

print("\n==================================================")
print("=== 2. Z-SCORE ANOMALY DETECTION (Spikes z > 2.5) ===")
print("==================================================")

# Compute z-score for technical decline percentage grouped by each bank
df['td_zscore'] = df.groupby('bank_name')['technical_decline_pct'].transform(
    lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1)
)

# Filter for severe anomalies where z-score exceeds 2.5 standard deviations
anomalies = df[df['td_zscore'] > 2.5][['month_year', 'bank_name', 'role_type', 'technical_decline_pct', 'td_zscore']]
print(f"Total severe operational anomaly months flagged (z > 2.5): {len(anomalies)}")
print(anomalies.head(10))


=== 2. Z-SCORE ANOMALY DETECTION (Spikes z > 2.5) ===
Total severe operational anomaly months flagged (z > 2.5): 123
    month_year                           bank_name role_type  \
0      2024-04                 State Bank Of India  Remitter   
11     2024-04                Airtel Payments Bank  Remitter   
18     2024-04                   IDBI Bank Limited  Remitter   
21     2024-04                        Yes Bank Ltd  Remitter   
40     2024-04     Rajasthan Marudhara Gramin Bank  Remitter   
45     2024-04        ESAF Small Finance Bank Ltd.  Remitter   
111    2026-04        Airtel Payments Bank Limited  Remitter   
137    2026-04  Ujjivan Small Finance Bank Limited  Remitter   
152    2024-08                      Bank of Baroda  Remitter   
153    2024-08                 Union Bank Of India  Remitter   

     technical_decline_pct  td_zscore  
0                     89.0   2.535229  
11                    72.0   2.543141  
18                    51.0   3.050015  
21               